**[Course Home](../README.md) | [Python Lab](../python/10_active_learning_and_experimental_design.ipynb) | [Prev: Ch 09: Rate-Distortion & VAEs](09_rate_distortion_and_information_bottleneck.ipynb) | [Next: Ch 11: Biological Information & Genomics](11_applied_genomics_and_neural_coding.ipynb)**

*Part V: Modern Frontiers: AI, Representation & Biology — Chapter 10 of 11*

---

# Chapter 10: Information in Action —
# Active Learning & Bayesian Experimental Design
### *Lindley's Expected Information Gain (EIG), Horstein Bisection, BALD, and*
### *Automated Culprit Localization*

---

## 1. Which Experiment Should You Run Next?

In classical passive machine learning, data collection is treated as an
external, solved problem. An engineer is handed a static table of
historical features and labels:
$$\mathcal{D} = \{(x_1, y_1), \dots, (x_N, y_N)\}$$
These samples are assumed to be drawn independently and identically
distributed (i.i.d.) from some fixed data distribution $p(x, y)$. The
practitioner's sole mandate is to fit an expressive model that minimizes
empirical risk over this static corpus.

Yet in the real world—from drug discovery and clinical pharmacology to
autonomous robotics and large-scale software systems—**data does not arrive
for free**.

### The Economics of Experimental Friction
In almost every high-consequence technical discipline, acquiring a single
ground-truth label $y$ incurs severe economic, temporal, computational, or
ethical costs:

* **Drug Discovery & Wet Labs**: A robotic high-throughput screening facility
  can synthesize and assay millions of candidate molecules against a viral
  protease. But each physical well plate requires expensive biochemical
  reagents, microfluidic consumables, and hours of incubation. Testing all
  $10^8$ commercially available small molecules would cost tens of millions
  of dollars and decades of laboratory throughput.
* **Clinical Dosage Trials**: In early-phase oncology trials, testing an
  experimental chemotherapy dosage level on a patient cohort is constrained
  by toxicity and human safety. Clinicians cannot sample dosage regimes at
  random; each trial cohort must balance prospective efficacy against the
  risk of adverse toxic reactions.
* **Automated Culprit Localization**: In hyperscale software monorepos, a
  broken change submitted somewhere within a sequence of 500 commits causes
  a subtle flaky integration test failure. Spinning up a virtualized
  hardware cluster to compile and execute the complete distributed test
  harness takes 45 minutes of server compute per commit. Testing every commit
  sequentially would stall release engineering pipelines for days.

### Passive Learning vs. Active Learning
Under severe resource constraints, passive observation is cripplingly
inefficient. Consider trying to discover the one-dimensional decision
boundary of a step function $f(x) = \mathbb{I}(x \ge \theta)$ on the unit
interval $x \in [0, 1]$:

* **Passive Random Sampling**: If you draw $N$ uniform random points
  $x_i \sim \text{Uniform}(0, 1)$ and observe their labels, the expected
  localization error of the threshold $\theta$ shrinks as $\mathcal{O}(1/N)$.
  Achieving an estimation precision of $\epsilon = 10^{-4}$ requires
  roughly $10,000$ costly evaluations.
* **Active Querying**: If an intelligent agent can choose *where* to sample
  next based on all prior observations, it can execute a bisection search,
  querying the midpoint of the remaining plausible interval. The uncertainty
  halves with every query! Achieving an accuracy of $\epsilon = 10^{-4}$
  requires only $\lceil \log_2(10^4) \rceil = 14$ queries.

The difference between $10,000$ experiments and $14$ experiments is not a
minor performance tweak—it represents a fundamental, exponential compression
of experimental effort.

```
                     THE ACTIVE LEARNING FEEDBACK LOOP
                     
            +-----------------------------------------------+
            |                                               |
            v                                               |
    +---------------+       +------------------+    +---------------+
    | Current Model | ----> |   Acquisition    | -> | Optimal Query |
    |  Belief p(θ)  |       |  Function (EIG)  |    |  x* = argmax  |
    +---------------+       +------------------+    +---------------+
            ^                                               |
            |                                               v
    +---------------+       +------------------+    +---------------+
    |   Bayesian    | <---- | Observed Outcome | <- |  Experiment / |
    |    Update     |       |        y*        |    | Lab / Sandbox |
    +---------------+       +------------------+    +---------------+
```

To build an agent capable of autonomous experimental inquiry, we require a
principled mathematical criterion for scoring prospective queries: **How do
we measure the information that an experiment has not yet delivered?**

---

## 2. Lindley's Information Utility & Expected Information Gain (EIG)

In 1956, British statistician Dennis V. Lindley published a landmark paper,
*"On a Measure of the Information Provided by an Experiment"*, providing a
rigorous foundation for Bayesian Experimental Design (BED) rooted in Claude
Shannon's information theory.

### The Bayesian Formulation
Let $\theta \in \Theta$ denote the unknown state of nature or latent parameter
vector we seek to identify (for example, the biochemical binding affinity of a
protein, the physical weight of an exoplanet, or the commit ID that broke a
build).

Prior to performing any new experiment, our current state of knowledge is
summarized by a prior probability distribution $p(\theta)$ with Shannon
entropy:
$$H(\theta) = -\int_{\Theta} p(\theta) \log_2 p(\theta) \, d\theta$$

Suppose we have a menu of candidate experiments or design decisions
$d \in \mathcal{D}$. If we conduct experiment $d$, nature generates an
observable measurement or outcome $Y \in \mathcal{Y}$ according to a known
likelihood model $p(y \mid \theta, d)$.

Before running experiment $d$, the prospective marginal distribution over
future outcomes $y$ is the prior predictive distribution:
$$p(y \mid d) = \int_{\Theta} p(y \mid \theta, d) p(\theta) \, d\theta$$

Once the experiment is executed and a specific measurement $y$ is observed,
our epistemic state updates via Bayes' rule to the posterior distribution:
$$p(\theta \mid y, d) = \frac{p(y \mid \theta, d) p(\theta)}{p(y \mid d)}$$

### Lindley's Information Utility
Lindley argued that the utility of observing an experimental outcome $y$ is
the gain in information about parameters $\theta$. In information theory,
the natural distance between prior belief $p(\theta)$ and updated belief
$p(\theta \mid y, d)$ is the Kullback-Leibler (KL) divergence:
$$U(d, y) = D_{\text{KL}}\big( p(\theta \mid y, d) \parallel p(\theta) \big)$$
$$= \int_{\Theta} p(\theta \mid y, d) \log_2 \left( \frac{p(\theta \mid y, d)}{p(\theta)} \right) d\theta$$

Because the experiment has not yet been executed, the outcome $y$ remains
unknown. Lindley defined the overall utility of an experiment $d$ as the
**Expected Information Gain (EIG)**, taking the expectation of this KL
divergence across all prospective outcomes weighted by predictive probability:

$$\text{EIG}(d) = \mathbb{E}_{Y \sim p(y \mid d)} \Big[ D_{\text{KL}}\big( p(\theta \mid Y, d) \parallel p(\theta) \big) \Big]$$

### The Fundamental Identity: EIG Is Mutual Information
Let us expand Lindley's expectation explicitly:
$$\text{EIG}(d) = \int_{\mathcal{Y}} p(y \mid d) \left[ \int_{\Theta} p(\theta \mid y, d) \log_2 \left( \frac{p(\theta \mid y, d)}{p(\theta)} \right) d\theta \right] dy$$

Using the product rule for joint distributions,
$p(\theta, y \mid d) = p(y \mid d) p(\theta \mid y, d)$:
$$\text{EIG}(d) = \int_{\mathcal{Y}} \int_{\Theta} p(\theta, y \mid d) \log_2 \left( \frac{p(\theta, y \mid d)}{p(\theta) p(y \mid d)} \right) d\theta \, dy$$

This double integral is none other than the **Mutual Information** between
latent hypothesis $\theta$ and observable measurement $Y$ under design $d$!

> **Theorem (Lindley 1956):**
> For any experiment $d$, the Expected Information Gain is identically equal
> to the conditional mutual information between parameter vector $\theta$
> and observable measurement $Y$:
> $$\text{EIG}(d) = I(\theta; Y \mid d)$$ 

### Two Complementary Views of Information Gain
By the fundamental symmetries of mutual information, $I(\theta; Y \mid d)$
can be expressed in two revealing ways:

#### View 1: The Parameter Space (Reduction in Epistemic Uncertainty)
$$I(\theta; Y \mid d) = H(\theta) - H(\theta \mid Y, d)$$ 
* $H(\theta)$ is our current prior uncertainty about the true hypothesis.
* $H(\theta \mid Y, d) = \mathbb{E}_{y}[H(\theta \mid y, d)]$ is the
  expected posterior uncertainty remaining after observing the outcome.
* **Design Goal**: Choose $d$ to make the expected posterior distribution as
  sharp, narrow, and peaked as possible.

#### View 2: The Measurement Space (Surprise vs. Inherent Noise)
$$I(\theta; Y \mid d) = H(Y \mid d) - H(Y \mid \theta, d)$$ 
* $H(Y \mid d)$ is the marginal entropy of the prospective observation.
  This measures how unpredictable the outcome is *a priori*.
* $H(Y \mid \theta, d) = \mathbb{E}_{\theta}[H(Y \mid \theta, d)]$ is the
  expected entropy of the measurement noise. This measures how unpredictable
  the outcome remains even if the true parameters are known.
* **Design Goal**: Choose $d$ so that the prospective outcome $Y$ has maximal
  marginal variance/entropy, while ensuring that the physical sensor noise
  is as small as possible.

```
                    THE INFORMATION GAIN BALANCE SHEET
                    
    +---------------------------------------------------------------+
    | Total Marginal Outcome Entropy: H(Y | d)                     |
    | (You want future observations to be globally unpredictable)   |
    +-------------------------------+-------------------------------+
    | Useful Information Gain:      | Inherent Sensor Noise:        |
    | EIG(d) = I(θ; Y | d)          | H(Y | θ, d)                   |
    | (Reduces uncertainty in θ)    | (Unavoidable physical fluff)  |
    +-------------------------------+-------------------------------+
```

### Asymptotic Connection: EIG and Fisher Information
When observations are continuous and the sample size $N$ is large, the
Bernstein-von Mises theorem guarantees that the posterior distribution
$p(\theta \mid y_1, \dots, y_N)$ converges to a multivariate normal
distribution whose covariance matrix is the inverse of the Fisher
Information matrix $\mathcal{I}(\theta)$ (established in Chapter 8).

Under a local Gaussian approximation, the differential entropy of a
Gaussian posterior with covariance $\Sigma$ is:
$$h(\theta \mid y) = \frac{1}{2} \log_2 \Big( (2\pi e)^k \det \Sigma \Big)$$
$$= -\frac{1}{2} \log_2 \det \mathcal{I}(\theta) + \text{const}$$

Minimizing posterior entropy is asymptotically equivalent to maximizing
$\log \det \mathcal{I}_d(\theta)$. This reveals that **Lindley's Bayesian
Information Criterion is the exact non-linear, non-asymptotic generalization
of classical D-Optimal Design** in experimental statistics!

---

## 3. Optimal Probing Strategies: 20 Questions & Horstein Bisection

How does Expected Information Gain translate into concrete, sequential
algorithms? The simplest and most famous realization is the classic parlor
game of **Twenty Questions**.

### The Formal 20 Questions Game
Suppose an opponent selects a secret object $X^*$ from a known dictionary
of $N$ candidate concepts $\mathcal{X} = \{x_1, x_2, \dots, x_N\}$.

1. **Initial Uncertainty**: Under a uniform prior $p(x) = 1/N$, the starting
   entropy is:
   $$H(X) = \log_2 N \text{ bits}$$
2. **A Query**: You ask a yes/no question $Q_k$ that partitions the candidate
   set into a subset $S \subset \mathcal{X}$ (where the answer is "Yes")
   and its complement $S^c$ (where the answer is "No").
3. **Entropy of the Answer**: Let $p = |S|/N$ be the prior probability of
   receiving a "Yes". The answer $Y \in \{\text{Yes}, \text{No}\}$ has
   binary entropy:
   $$H(Y) = h_2(p) = -p \log_2 p - (1-p) \log_2(1-p) \le 1.0 \text{ bit}$$
4. **Information Transferred**: Because the true object deterministically
   dictates the answer, $H(Y \mid X) = 0$. Therefore:
   $$I(X; Y) = H(Y) - H(Y \mid X) = H(Y) = h_2(p)$$ 

To maximize information gain at every turn, you must choose a question where
$p = 1/2$. A query that splits the hypothesis space exactly in half yields
identically **1.0 bit of information**.

With 20 perfectly balanced queries, an interrogator extracts $20$ bits of
information, isolating 1 specific concept among:
$$N = 2^{20} = 1,048,576 \text{ possibilities!}$$

### The Fragility of Classical Binary Search Under Noise
Classical binary search algorithms (such as the standard `git bisect`
command used by software engineers) are deterministic 20-Questions
strategies. They assume that every query observation is **noise-free**.

In the real world, tests are noisy:
* A flaky integration test might pass $15\%$ of the time on a broken commit
  due to a network race condition.
* A medical diagnostic test produces false negatives with probability
  $\epsilon$.
* A crowd-sourced human annotator misreads instructions and enters the
  wrong label.

If you execute classical binary search over a noisy channel, disaster strikes:
1. The search algorithm tests a commit at the midpoint.
2. The commit is buggy, but due to a flaky false pass, the test returns
   `PASS`.
3. Classical bisection discards the entire left half of the commit history.
4. **The true culprit has been permanently eliminated.** Every subsequent
   bisection step searches deeper into the wrong branch.

```
                   THE CATASTROPHE OF NOISY BINARY SEARCH
                   
    Commits: [ C0, C1, C2, C3, C4, C5, C6, C7 ]   (True Culprit = C2)
                               ^
                          Test C4 -> [FAIL]
               Subtree: [ C0, C1, C2, C3 ]
                          ^
                     Test C2 -> [FALSE PASS!] (15% flake)
                                     |
               Subtree: [ C3 ] <-----+ (C0, C1, C2 DISCARDED FOREVER!)
                     Deterministic binary search terminates on C3.
                     RESULT: Completely wrong root cause identified!
```

### Horstein's Probabilistic Bisection Algorithm
In 1963, Michael Horstein solved this problem by designing the optimal
information-theoretic bisection algorithm for communication over a Binary
Symmetric Channel (BSC) with crossover probability $\epsilon \in (0, 0.5)$.

Instead of maintaining a hard interval and discarding candidate commits,
Horstein's algorithm maintains a **smooth probability distribution** $p_t(c)$
over all candidate culprit locations $c \in [0, 1]$.

#### Step 1: Query the Posterior Median
At step $t$, compute the cumulative distribution function (CDF):
$$F_t(x) = \int_0^x p_t(\theta) \, d\theta$$
Find the median point $x_t^*$ where exactly half of the current probability
mass lies to the left and half lies to the right:
$$F_t(x_t^*) = 0.5$$

Because the median point splits belief equally, the probability of observing
a positive or negative test outcome is as close to $50/50$ as possible,
ensuring that prospective outcome entropy $H(Y)$ is maximized!

#### Step 2: Smooth Bayesian Likelihood Update
We query test point $x_t^*$ and observe a noisy outcome $y_t \in \{0, 1\}$.
Instead of discarding any region, we apply Bayes' rule across the entire
domain:

* If the test reports $y_t = 0$ (indicating bug is left of $x_t^*$):
  $$p_{t+1}(\theta) \propto \begin{cases}
  (1 - \epsilon) \cdot p_t(\theta) & \text{if } \theta \le x_t^* \\
  \epsilon \cdot p_t(\theta) & \text{if } \theta > x_t^*
  \end{cases}$$
* If the test reports $y_t = 1$ (indicating bug is right of $x_t^*$):
  $$p_{t+1}(\theta) \propto \begin{cases}
  \epsilon \cdot p_t(\theta) & \text{if } \theta \le x_t^* \\
  (1 - \epsilon) \cdot p_t(\theta) & \text{if } \theta > x_t^*
  \end{cases}$$

Normalize $p_{t+1}(\theta)$ so it integrates to 1.0.

```
                POSTERIOR EVOLUTION IN HORSTEIN BISECTION
                
    p_0(θ) | ------------------------------------- (Uniform Prior, H = 3 bits)
           +------------------+------------------+
           0                 0.5                 1.0
                              ^ Query Median
                              |
    p_1(θ) |       /\                            (After observation y_1)
           |      /  \                           (Smooth probability shift)
           +-----+----+--------------------------+
           0    0.25 0.5                         1.0
                              |
    p_K(θ) |           ||                        (After K noisy queries)
           |           ||                        (Concentrates into sharp
           +-----------++------------------------+  Dirac peak at true culprit)
           0           θ*                        1.0
```

#### The Information-Theoretic Rate of Convergence
What is the expected information gain of Horstein bisection under noise
$\epsilon$? Recall from Chapter 5 that a Binary Symmetric Channel with error
rate $\epsilon$ has channel capacity:
$$C = 1 - h_2(\epsilon) \text{ bits/query}$$
$$= 1 + \epsilon \log_2 \epsilon + (1-\epsilon) \log_2(1-\epsilon)$$

Horstein proved that querying the posterior median achieves the channel
capacity at every step! On average, each noisy test query reduces the
remaining uncertainty by exactly $C$ bits:
$$\mathbb{E}[H(p_{t+1})] = H(p_t) - \big( 1 - h_2(\epsilon) \big)$$

Even if test flakiness is high (say $\epsilon = 0.20$, so $C = 0.278$
bits/query), Horstein bisection is mathematically guaranteed to converge
to the true culprit. It never gets trapped down a false branch because
every region retains non-zero posterior mass!

---

## 4. Active Learning in Modern Machine Learning: Uncertainty Sampling vs. BALD

In modern supervised deep learning, active learning is typically deployed
in a **pool-based** setting:
1. We have a small labeled training dataset $\mathcal{L} = \{(x_i, y_i)\}$.
2. We have a massive unlabeled pool $\mathcal{U} = \{x_j\}$.
3. At each iteration, an acquisition function $\alpha(x)$ scores every
   candidate sample in $\mathcal{U}$. We select the top sample to send to
   a human annotator for ground-truth labeling.

### Heuristic Uncertainty Sampling
The simplest and most widely used baseline is **uncertainty sampling**,
which queries instances where the current model is most unsure about its
prediction:

* **Least Confidence**: Query the sample where the model's highest predicted
  class probability is lowest:
  $$\alpha_{\text{LC}}(x) = 1 - \max_c p(y=c \mid x, \mathcal{D})$$
* **Margin Sampling**: Query the sample where the difference between the top
  two most probable classes is smallest:
  $$\alpha_{\text{Margin}}(x) = -\big( p(y = \hat{c}_1 \mid x) - p(y = \hat{c}_2 \mid x) \big)$$ 
* **Predictive Entropy**: Query the sample where the Shannon entropy of the
  model's predictive label distribution is maximized:
  $$\alpha_{\text{Entropy}}(x) = H(y \mid x, \mathcal{D})$$
  $$= -\sum_{c} p(y=c \mid x, \mathcal{D}) \log_2 p(y=c \mid x, \mathcal{D})$$

While intuitive, heuristic uncertainty sampling suffers from a fatal
theoretical blind spot: **the Aleatoric Uncertainty Trap**.

### Epistemic vs. Aleatoric Uncertainty: The Fatal Flaw
To understand why predictive entropy fails in real applications, we must
dissect the two fundamentally distinct forms of uncertainty in machine
learning:

* **Epistemic Uncertainty (Model Ignorance)**: Arises from a lack of
  training data in a specific region of feature space. It is reducible with
  more data. Example: an image of a rare animal species never encountered
  in the training set.
* **Aleatoric Uncertainty (Data Noise)**: Arises from inherent physical
  stochasticity, sensor corruption, or class overlap. It is irreducible by
  collecting more samples. Example: a completely corrupted, out-of-focus image
  or a fair coin flip.

Predictive entropy $H(y \mid x, \mathcal{D})$ measures **total** uncertainty.
It cannot distinguish whether a sample is uncertain because the model lacks
knowledge (epistemic) or because the label is fundamentally random (aleatoric).

> **The Trap**: Suppose your unlabeled pool contains corrupted images
> consisting of pure static noise or images with overlapping labels. For such
> images, the model predicts $p(y=1 \mid x) = 0.5$, yielding a predictive
> entropy of $1.0$ bit!
> 
> An active learner driven by naive predictive entropy will repeatedly spend
> its entire annotation budget querying meaningless static noise, learning
> exactly zero new information about the underlying classification boundary!

### Bayesian Active Learning by Disagreement (BALD)
In 2011, Neil Houlsby, Ferenc Huszar, Zoubin Ghahramani, and Jose Miguel
Hernandez-Lobato introduced **BALD (Bayesian Active Learning by Disagreement)**,
subsequently extended to deep neural networks via Monte Carlo Dropout by
Yarin Gal, Riashat Islam, and Zoubin Ghahramani (2017).

BALD applies Lindley's information utility directly to the parameters of the
machine learning model. Instead of asking *"Which label is most uncertain?"*,
BALD asks:
> *"Which unlabeled candidate $x$ maximizes the mutual information between the
> candidate's label $y$ and the model's weight vector $\mathbf{w}$?"*

Using the mutual information identity:
$$\text{BALD}(x) = I(y; \mathbf{w} \mid x, \mathcal{D})$$
$$= H(y \mid x, \mathcal{D}) - \mathbb{E}_{\mathbf{w} \sim p(\mathbf{w} \mid \mathcal{D})} \big[ H(y \mid x, \mathbf{w}) \big]$$

Let us dissect this equation term by term:

1. **First Term: Total Predictive Entropy** $H(y \mid x, \mathcal{D})$
   This is the entropy of the marginal prediction averaged over the entire
   posterior ensemble of plausible parameter configurations $\mathbf{w}$:
   $$p(y \mid x, \mathcal{D}) = \int p(y \mid x, \mathbf{w}) p(\mathbf{w} \mid \mathcal{D}) \, d\mathbf{w}$$
2. **Second Term: Expected Aleatoric Noise**
   $$\mathbb{E}_{\mathbf{w} \sim p(\mathbf{w} \mid \mathcal{D})}[H(y \mid x, \mathbf{w})]$$
   For any specific fixed model parameter vector $\mathbf{w}$,
   $H(y \mid x, \mathbf{w})$ is the entropy of the prediction. This measures
   the inherent noise of the data conditioned on knowing the true model.
3. **The Difference: Epistemic Disagreement**
   BALD subtracts the expected aleatoric noise from total entropy. What
   remains is **pure epistemic uncertainty**—the extent to which different
   plausible models drawn from the posterior *disagree* on the label of $x$!

```
                     HOW BALD ESCAPES THE NOISE TRAP
                     
    Scenario A: Pure Noise (Corrupted Static)
    - Model 1 predicts: p(y=1) = 0.5 -> H = 1.0 bit
    - Model 2 predicts: p(y=1) = 0.5 -> H = 1.0 bit
    - Ensemble Average: p(y=1) = 0.5 -> Total H = 1.0 bit
    ==> BALD Score = 1.0 - 1.0 = 0.0 bits (IGNORED! NO ANNOTATION WASTED)
    
    Scenario B: High Epistemic Ambiguity (Unexplored Feature Region)
    - Model 1 predicts: p(y=1) = 0.99 -> H = 0.08 bits (Confident YES)
    - Model 2 predicts: p(y=1) = 0.01 -> H = 0.08 bits (Confident NO)
    - Ensemble Average: p(y=1) = 0.50 -> Total H = 1.00 bits
    ==> BALD Score = 1.00 - 0.08 = 0.92 bits (MAXIMAL ACQUISITION PRIORITY!)
```

By subtracting internal model uncertainty, BALD achieves complete immunity to
aleatoric noise, isolating samples where acquiring a label genuinely teaches
the model new parameters.

---

## 5. Applied Engineering Contexts

The mathematical machinery of Expected Information Gain is not restricted to
academic benchmarks. It drives critical infrastructure in production systems.

### 1. Automated Debugging in Complex Microservice Graphs
In distributed architectures comprising thousands of microservices
communicating via remote procedure calls (RPCs), pinpointing the root cause
of an end-to-end latency spike or error condition is a major operational
challenge.

* **The Naive Approach**: Log every single packet or trace every RPC. At
  hyperscale, full distributed tracing generates petabytes of telemetry per
  second, swamping storage networks.
* **Information-Theoretic Probing**: Treat the failure culprit as a random
  variable $S^* \in \{S_1, \dots, S_M\}$ over the directed acyclic graph
  (DAG) of services. Placing a diagnostic probe or synthetic health check on
  an internal network edge $e = (u, v)$ provides an observation $Y_e$.
* **Cost-Weighted EIG**: Diagnostic probes have differing latency and network
  costs. The optimal automated triage system maximizes the information
  yield per unit cost:
  $$\text{Utility}(e) = \frac{I(S^*; Y_e)}{\text{Cost}(e)}$$
  Probing the minimum-cost graph cut that bisects the posterior probability
  mass isolates the failing service in logarithmic time with minimal
  diagnostic overhead.

### 2. A/B Testing: Multi-Armed Bandits vs. Pure Exploration (EIG)
A frequent source of confusion in industrial data science is the distinction
between **Multi-Armed Bandits** and **Active Learning (Bayesian Experimental
Design)**.

```
                     BANDITS VS. ACTIVE LEARNING
                     
          Bandits (Thompson / UCB)            Active Learning (EIG / BALD)
       +----------------------------------+----------------------------------+
       | Objective: Cumulative Regret     | Objective: Simple Regret         |
       | Minimize losses incurred DURING  | Find the true parameter θ with   |
       | the experiment itself.           | maximum confidence for LATER use.|
       +----------------------------------+----------------------------------+
       | Policy: Exploit high-performing  | Policy: Query maximum uncertainty|
       | arms while exploring promising.  | regions regardless of return.    |
       +----------------------------------+----------------------------------+
       | Application: Ad click-through    | Application: Clinical drug safety|
       | optimization, website conversion.| aerodynamic stress testing.      |
       +----------------------------------+----------------------------------+
```

* **Use a Bandit** when every query has an immediate business payout or
  penalty (e.g., serving search advertisements). You cannot afford to serve
  a terrible ad solely to learn that users hate it.
* **Use Bayesian Experimental Design (EIG)** when you are in an offline or
  pre-deployment phase where the ultimate goal is accurate parameter
  identification (e.g., calibrating aircraft sensor flight software or
  finding a subtle concurrency race condition in a staging environment).
  Here, acquiring pure information is the sole and sovereign objective.

---

**[Course Home](../README.md) | [Python Lab](../python/10_active_learning_and_experimental_design.ipynb) | [Prev: Ch 09: Rate-Distortion & VAEs](09_rate_distortion_and_information_bottleneck.ipynb) | [Next: Ch 11: Biological Information & Genomics](11_applied_genomics_and_neural_coding.ipynb)**